# TF-IDF Pipeline

Firstly we will load the data files and create the TF-IDF vector, and afterwards we train a simple logistic regression classifier to later compare with the GNN approach

In [21]:
import pandas as pd
import gzip
import json
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

In [ ]:
def parse_json_to_dict(path):
    with gzip.open(path, 'rb') as f:
        for line in f:
            yield json.loads(line)

def load_to_df(path, max_items=10000):
    data = []
    for i, entry in enumerate(parse_json_to_dict(path)):
        if i >= max_items: 
            break
        
        # here we could include more stuff from the review, but for now
        # lets use just the text
        text = entry.get('reviewText', '')
        
        # temporary way to get category from filename
        category = path.split('/')[-1].split('.')[0] 
        
        data.append({
            'text': f"{text}".strip(),
            'category': category
        })
    return pd.DataFrame(data)

In [ ]:
music_instruments_df = load_to_df('data/musical_instruments.json.gz')
automotive_df = load_to_df('data/automotive.json.gz')

# add the others here ...

df = pd.concat([
    music_instruments_df, automotive_df
], ignore_index=True)

In [ ]:
vectorizer = TfidfVectorizer(stop_words='english', max_features=5000)
X = vectorizer.fit_transform(df['text'])
y = df['category']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
clf = LogisticRegression(random_state=42)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, y_pred)}")
print(classification_report(y_test, y_pred))

Accuracy: 0.95575
                     precision    recall  f1-score   support

         automotive       0.94      0.98      0.96      1981
musical_instruments       0.98      0.94      0.96      2019

           accuracy                           0.96      4000
          macro avg       0.96      0.96      0.96      4000
       weighted avg       0.96      0.96      0.96      4000



In [29]:
mock_reviews = [
    "This guitar has a great sound and is perfect for beginners.",
    "This truck has a great suspension and is perfect for beginners."
]

mock_X = vectorizer.transform(mock_reviews)
mock_pred = clf.predict(mock_X)
for review, pred in zip(mock_reviews, mock_pred):
    print(f"Review: {review}\nPredicted Category: {pred}\n")

Review: This guitar has a great sound and is perfect for beginners.
Predicted Category: musical_instruments

Review: This truck has a great suspension and is perfect for beginners.
Predicted Category: automotive

